<a href="https://colab.research.google.com/github/winicius87/NBA-2026-Prediction-with-Tensorflow/blob/main/NBA_2026_Prediction_with_Tensorflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'''''import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize pixel values (0–255) to (0–1)
x_train, x_test = x_train / 255.0, x_test / 255.0
model = keras.Sequential([
      layers.Flatten(input_shape=(28, 28)),   # Converts 2D image to 1D
          layers.Dense(128, activation='relu'),   # Hidden layer
              layers.Dropout(0.2),                    # Prevent overfitting
                  layers.Dense(10, activation='softmax')  # Output layer
                  ])
model.compile(
      optimizer='adam',
          loss='sparse_categorical_crossentropy',
              metrics=['accuracy']
              )

model.fit(x_train, y_train, epochs=5)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print('\nTest accuracy:', test_acc)

# Save model
model.save('my_first_nn.h5')

# Load model
loaded_model = keras.models.load_model('my_first_nn.h5')


Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.9104 - loss: 0.3034
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.9563 - loss: 0.1472
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9664 - loss: 0.1109
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9723 - loss: 0.0905
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9767 - loss: 0.0742
313/313 - 1s - 2ms/step - accuracy: 0.9757 - loss: 0.0785

Test accuracy: 0.9757000207901001


In [ ]:
import tensorflow as tf
import numpy as np

# 1. Generate sample 3D data points lying on a plane (e.g., z = 2x - 3y + 5)
np.random.seed(42)
x_data = np.random.uniform(-10, 10, 100)
y_data = np.random.uniform(-10, 10, 100)
# Add some noise to the data
z_data = 2 * x_data - 3 * y_data + 5 + np.random.normal(0, 0.1, 100)

# 2. Combine X and Y into a single input matrix (Shape: 100 samples, 2 features)
xy_data = np.column_stack((x_data, y_data))

# 3. Define the single-layer neural network (no activation = linear model)
# Dense(1) inherently represents: Output = (Weight_X * X) + (Weight_Y * Y) + Bias
model = tf.keras.Sequential([
    tf.keras.layers.Dense(units=1, input_shape=(2,))
    ])

# 4. Compile model with Mean Squared Error loss and an optimizer
model.compile(optimizer='adam', loss='mean_squared_error')

# 5. Train the model to fit the plane
model.fit(xy_data, z_data, epochs=5000, verbose=0)

# 6. Extract the learned parameters
weights, bias = model.layers[0].get_weights()
a, b = weights[0][0], weights[1][0]
c = bias[0]

print(f"Learned Plane Equation: z = {a:.4f}x + {b:.4f}y + {c:.4f}")

Learned Plane Equation: z = 1.9983x + -2.9964y + 5.0100


In [ ]:
from google.colab import auth
import gspread
from google.auth import default

# 1. Authenticate your Google Account
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 2. Open the Google Sheets file by its exact name
spreadsheet = gc.open("Sasvsknicks")

# 3. Open the second sheet (tab)
# Note: get_worksheet(1) uses a 0-based index (0=first sheet, 1=second sheet)
worksheet = spreadsheet.get_worksheet(1) # Renamed to worksheet for clarity

# 4. Extract data and convert to TensorFlow compatible format (e.g., Pandas DataFrame)
import pandas as pd
raw_data = worksheet.get_all_values() # Get all data as list of lists

# Assuming the first row is the header
headers = raw_data[0]
data_rows = raw_data[1:]

# Generate unique column names to handle potential duplicates in raw_data[0]
seen_headers = {}
unique_headers = []
for h in headers:
    original_h = h
    counter = 1
    while h in seen_headers:
        h = f"{original_h}_{counter}"
        counter += 1
    seen_headers[h] = True
    unique_headers.append(h)

df_pandas = pd.DataFrame(data_rows, columns=unique_headers)

# Convert the target column (first column) to numerical (0 or 1)
# Assuming 'W' maps to 1 and 'L' maps to 0
df_pandas.iloc[:, 1] = df_pandas.iloc[:, 1].apply(lambda x: 1 if str(x).strip().upper() == 'W' else (0 if str(x).strip().upper() == 'L' else pd.NA))

# Convert the rest of the columns (features) to numeric
for col in df_pandas.columns[2:]:
    df_pandas[col] = pd.to_numeric(df_pandas[col], errors='coerce')

# Drop any rows that resulted in NaN after conversion (e.g., empty cells, non-numeric strings in numeric columns)
df_pandas.dropna(inplace=True)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load data - already loaded into df_pandas

n_predict = 2

# 2. Separate Target (1st column) and Features
y = df_pandas.iloc[:-n_predict, 1].values.astype(int)  # First column is your binary outcome (0 or 1)
X = df_pandas.iloc[:-n_predict, 2:].values  # Remaining columns are your features
pred = df_pandas.iloc[-n_predict:, 2:].values  # Remaining columns are your features


# 3. Scale the features (Neural networks perform best on scaled data)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pred_scaled = scaler.fit_transform(pred)

#X_scaled = X_scaled0.iloc[:-n_predict]
#X_predict = X_scaled0.iloc[-n_predict:]


# 4. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


import tensorflow as tf
from tensorflow.keras import layers, Sequential

# Define the number of features
num_features = X.shape[1]

model = Sequential([
    layers.Dense(64, activation='relu', input_shape=(num_features,)),
        layers.Dense(32, activation='relu'),
            layers.Dense(1, activation='sigmoid')  # 1 output neuron with Sigmoid activation for binary [0, 1] outcome
            ])
model.compile(optimizer='adam',
              loss='binary_crossentropy',
                            metrics=['accuracy'])
# Train the model
history = model.fit(X_train, y_train,
                    epochs=20,
                                        batch_size=32,
                                                            validation_split=0.2)

# Evaluate on test data
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_accuracy:.2f}")

# Predict probabilities (e.g., 0.85 chance of being Class 1)
y_pred_probs = model.predict(X_test)
# Convert probabilities to binary 0 or 1
y_pred_classes = (y_pred_probs >= 0.5).astype(int)

pred_probs = model.predict(pred_scaled)
pred_classes = (pred_probs >= 0.5).astype(int)


print(f"Predictions : {pred_probs}")









Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.4737 - loss: 0.6953 - val_accuracy: 0.4000 - val_loss: 0.6469
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - accuracy: 0.5263 - loss: 0.6600 - val_accuracy: 0.8000 - val_loss: 0.6247
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.5263 - loss: 0.6269 - val_accuracy: 0.8000 - val_loss: 0.6034
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.7368 - loss: 0.5968 - val_accuracy: 0.8000 - val_loss: 0.5829
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - accuracy: 0.7895 - loss: 0.5693 - val_accuracy: 0.8000 - val_loss: 0.5632
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.7895 - loss: 0.5442 - val_accuracy: 1.0000 - val_loss: 0.5445
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - accuracy: 0.8947 - loss: 0.5206 - val_accuracy: 1.0000 - val_loss: 0.5267
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - accuracy: 0.8947 - loss: 0.4982 - val_accuracy: 1.0000 - val_loss: 0.5098
Epoch 9/20
